# Task 2. RPC Provider 연결과 Python 설정 검증

## 검증 의도

이 노트북은 `.env`가 현재 Python 소스 코드의 설정 객체와 실제 RPC client까지 연결되는지 확인합니다.

직접 검증하는 코드는 `src/cryptoquant_pipeline/config.py`, `provider.py`, `rpc_client.py`입니다.

## 검증 경계

- secret, provider URL 원문, API key는 출력하지 않습니다.
- `ETH_RPC_URL` 또는 legacy `CHAINSTACK_RPC_URL`이 없으면 `BLOCKED`로 표시합니다.
- `eth_chainId`, `eth_getBlockByNumber("finalized")` smoke 결과만 확인합니다.
- 1시간 전체 수집과 backfill은 이 노트북에서 수행하지 않습니다.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    """노트북 실행 위치와 무관하게 현재 repository root를 찾는다."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "cryptoquant_pipeline").exists():
            return candidate
    raise RuntimeError("repository root를 찾지 못함. pyproject.toml과 src/cryptoquant_pipeline 확인 필요함.")


PROJECT_ROOT = find_repo_root()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

ENV_PATH = PROJECT_ROOT / ".env"
if ENV_PATH.exists():
    load_dotenv(ENV_PATH)

SAFE_ENV_STATUS = {
    "project_root": str(PROJECT_ROOT),
    "env_file_exists": ENV_PATH.exists(),
    "eth_rpc_url_configured": bool(os.getenv("ETH_RPC_URL") or os.getenv("CHAINSTACK_RPC_URL")),
    "auth_mode": os.getenv("ETH_RPC_AUTH_MODE") or os.getenv("CHAINSTACK_AUTH_MODE") or "none",
}

display(Markdown("### Repository 및 환경변수 확인"))
display(SAFE_ENV_STATUS)

### Repository 및 환경변수 확인

{'project_root': '/workspace',
 'env_file_exists': True,
 'eth_rpc_url_configured': True,
 'auth_mode': 'none'}

In [2]:
from cryptoquant_pipeline.config import PipelineSettings
from cryptoquant_pipeline.exceptions import ConfigError

settings = None
settings_status = "READY"
settings_error = None
try:
    settings = PipelineSettings.from_env()
except ConfigError as exc:
    settings_status = "BLOCKED"
    settings_error = f"{exc.__class__.__name__}: {exc}"

safe_settings = {
    "status": settings_status,
    "error": settings_error,
    "chain_id": None if settings is None else settings.chain_id,
    "auth_mode": None if settings is None else settings.provider.auth_mode,
    "provider_configured": settings is not None,
    "max_blocks_per_log_request": None if settings is None else settings.max_blocks_per_log_request,
    "requests_per_second": None if settings is None else settings.rpc_requests_per_second,
    "delta_logs_path": None if settings is None else str(settings.delta_logs_path),
    "duckdb_path": None if settings is None else str(settings.duckdb_path),
    "collection_scope": None if settings is None else settings.collection_scope.scope_id,
    "collection_scope_fingerprint": None if settings is None else settings.collection_scope.fingerprint,
}

display(Markdown("### Python 설정 객체 검증"))
display(safe_settings)

### Python 설정 객체 검증

{'status': 'READY',
 'error': None,
 'chain_id': 1,
 'auth_mode': 'none',
 'provider_configured': True,
 'max_blocks_per_log_request': 10,
 'requests_per_second': 4.0,
 'delta_logs_path': '/opt/airflow/data/delta/ethereum_logs_v2',
 'duckdb_path': '/opt/airflow/data/analytics/ethereum_analytics_v2.duckdb',
 'collection_scope': 'transfer_topic_all_addresses',
 'collection_scope_fingerprint': 'af0864e874141de6657364a36407791a0769dee5c55679b2089c4efb33e8b885'}

In [3]:
from cryptoquant_pipeline.rpc_client import EthereumJsonRpcClient

if settings is None:
    display(Markdown("### RPC smoke skipped\n`ETH_RPC_URL`이 없어 실제 provider 호출을 실행하지 않았습니다."))
else:
    with EthereumJsonRpcClient(
        settings.provider,
        timeout_seconds=settings.rpc_timeout_seconds,
        max_retries=settings.rpc_max_retries,
        requests_per_second=settings.rpc_requests_per_second,
    ) as client:
        chain_id = client.eth_chain_id()
        finalized_block = client.eth_get_finalized_block()

    finalized_summary = {
        "chain_id": chain_id,
        "expected_chain_id": settings.chain_id,
        "chain_id_status": "PASS" if chain_id == settings.chain_id else "FAIL",
        "finalized_block_number": int(finalized_block["number"], 16),
        "finalized_block_hash_prefix": finalized_block["hash"][:18] + "...",
        "finalized_block_timestamp_utc_epoch": int(finalized_block["timestamp"], 16),
    }
    display(Markdown("### RPC smoke 결과"))
    display(finalized_summary)
    if chain_id != settings.chain_id:
        raise RuntimeError("provider chain id가 ETH_CHAIN_ID와 다름.")

### RPC smoke 결과

{'chain_id': 1,
 'expected_chain_id': 1,
 'chain_id_status': 'PASS',
 'finalized_block_number': 25374023,
 'finalized_block_hash_prefix': '0xf7d8b61a74ddcd7f...',
 'finalized_block_timestamp_utc_epoch': 1782141911}